# Trader Performance vs Market Sentiment


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 1) Loading the trader dataset to inspect it

In [3]:
df1 = pd.read_csv("historical_data.csv")

print("Shape:", df1.shape)
print("\nColumns:")
print(df1.columns.tolist())

print("\nNull values:")
print(df1.isna().sum().sort_values(ascending=False))

print("\nExact duplicate rows:", df1.duplicated().sum())


Shape: (211224, 16)

Columns:
['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']

Null values:
Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64

Exact duplicate rows: 0


## 2) Converting the trader timestamp to create a clean daily `date`


In [4]:
df1["Timestamp IST"] = pd.to_datetime(df1["Timestamp IST"], format="%d-%m-%Y %H:%M", errors="coerce")
df1["date"] = df1["Timestamp IST"].dt.normalize()

# Drop rows with an invalid timestamp, just in case
df1 = df1.dropna(subset=["date"]).copy()

print(df1[["Timestamp IST", "date"]].head())
print("\nTrader date range:", df1["date"].min(), "to", df1["date"].max())
print("Unique trader dates:", df1["date"].nunique())


        Timestamp IST       date
0 2024-12-02 22:50:00 2024-12-02
1 2024-12-02 22:50:00 2024-12-02
2 2024-12-02 22:50:00 2024-12-02
3 2024-12-02 22:50:00 2024-12-02
4 2024-12-02 22:50:00 2024-12-02

Trader date range: 2023-05-01 00:00:00 to 2025-05-01 00:00:00
Unique trader dates: 480


## 3) Loading the Fear/Greed sentiment dataset to inspect it

In [5]:
df2 = pd.read_csv("fear_greed_index.csv")

print("Shape:", df2.shape)
print("\nColumns:")
print(df2.columns.tolist())

print("\nNull values:")
print(df2.isna().sum().sort_values(ascending=False))

print("\nExact duplicate rows:", df2.duplicated().sum())


Shape: (2644, 4)

Columns:
['timestamp', 'value', 'classification', 'date']

Null values:
timestamp         0
value             0
classification    0
date              0
dtype: int64

Exact duplicate rows: 0


## 4) Converting the sentiment date column


In [6]:
df2["date"] = pd.to_datetime(df2["date"], errors="coerce").dt.normalize()
df2 = df2.dropna(subset=["date"]).copy()

# The timestamp column is redundant for a daily merge
if "timestamp" in df2.columns:
    df2 = df2.drop(columns=["timestamp"])

print(df2[["date", "classification", "value"]].head())
print("\nSentiment date range:", df2["date"].min(), "to", df2["date"].max())
print("Unique sentiment dates:", df2["date"].nunique())


        date classification  value
0 2018-02-01           Fear     30
1 2018-02-02   Extreme Fear     15
2 2018-02-03           Fear     40
3 2018-02-04   Extreme Fear     24
4 2018-02-05   Extreme Fear     11

Sentiment date range: 2018-02-01 00:00:00 to 2025-05-02 00:00:00
Unique sentiment dates: 2644


## 5) Building key trader features


In [7]:
# Helper flags
df1["is_win"] = (df1["Closed PnL"] > 0).astype(int)
df1["is_long"] = df1["Direction"].astype(str).str.contains("Long", case=False, na=False).astype(int)
df1["is_short"] = df1["Direction"].astype(str).str.contains("Short", case=False, na=False).astype(int)

# Core daily trader features
daily_features = df1.groupby(["Account", "date"]).agg(
    daily_pnl=("Closed PnL", "sum"),
    trades_per_day=("Trade ID", "count"),
    avg_trade_size=("Size USD", "mean"),
    win_rate=("is_win", "mean"),
).reset_index()

# Directional bias with smoothing to avoid division-by-zero NaNs
direction_stats = df1.groupby(["Account", "date"]).agg(
    long_trades=("is_long", "sum"),
    short_trades=("is_short", "sum"),
).reset_index()

direction_stats["long_short_ratio"] = (direction_stats["long_trades"] + 1) / (direction_stats["short_trades"] + 1)

daily_features = daily_features.merge(
    direction_stats[["Account", "date", "long_short_ratio"]],
    on=["Account", "date"],
    how="left"
)

print(daily_features.head())
print("\nDaily feature shape:", daily_features.shape)
print("Missing values in daily_features:\n", daily_features.isna().sum())


                                      Account       date  daily_pnl  trades_per_day  avg_trade_size  win_rate  \
0  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-11        0.0             177     5089.718249  0.000000   
1  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-17        0.0              68     7976.664412  0.000000   
2  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-18        0.0              40    23734.500000  0.000000   
3  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-22   -21227.0              12    28186.666667  0.000000   
4  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-26     1603.1              27    17248.148148  0.444444   

   long_short_ratio  
0          0.005618  
1          0.014493  
2          0.024390  
3          0.076923  
4          0.035714  

Daily feature shape: (2341, 7)
Missing values in daily_features:
 Account             0
date                0
daily_pnl           0
trades_per_day      0
avg_trade_size      0
win_rate    

## 6) Merging daily trader features with sentiment

In [8]:
df = daily_features.merge(
    df2[["date", "classification", "value"]],
    on="date",
    how="left"
)

print(df.head())
print("\nMerged shape:", df.shape)
print("\nMissing values after merge:")
print(df.isna().sum().sort_values(ascending=False))


                                      Account       date  daily_pnl  trades_per_day  avg_trade_size  win_rate  \
0  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-11        0.0             177     5089.718249  0.000000   
1  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-17        0.0              68     7976.664412  0.000000   
2  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-18        0.0              40    23734.500000  0.000000   
3  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-22   -21227.0              12    28186.666667  0.000000   
4  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-26     1603.1              27    17248.148148  0.444444   

   long_short_ratio classification  value  
0          0.005618  Extreme Greed   76.0  
1          0.014493  Extreme Greed   90.0  
2          0.024390  Extreme Greed   83.0  
3          0.076923  Extreme Greed   94.0  
4          0.035714  Extreme Greed   79.0  

Merged shape: (2341, 9)

Missing values after merge:
cla

## 7) Removing unmatched rows and dropping redundant columns


In [9]:
# Drop rows without sentiment labels
df = df.dropna(subset=["classification", "value"]).copy()

# Final clean column order
df = df[[
    "Account",
    "date",
    "daily_pnl",
    "trades_per_day",
    "avg_trade_size",
    "win_rate",
    "long_short_ratio",
    "classification",
    "value"
]].copy()

# Optional: sort for readability
df = df.sort_values(["Account", "date"]).reset_index(drop=True)

print("Final cleaned merged shape:", df.shape)
print("\nFinal missing values:")
print(df.isna().sum())

df.head()


Final cleaned merged shape: (2340, 9)

Final missing values:
Account             0
date                0
daily_pnl           0
trades_per_day      0
avg_trade_size      0
win_rate            0
long_short_ratio    0
classification      0
value               0
dtype: int64


,Account,date,daily_pnl,trades_per_day,avg_trade_size,win_rate,long_short_ratio,classification,value
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,0.0,177,5089.718249,0.000000,0.005618,Extreme Greed,76.0
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,0.0,68,7976.664412,0.000000,0.014493,Extreme Greed,90.0
2,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-18,0.0,40,23734.500000,0.000000,0.024390,Extreme Greed,83.0
3,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-22,-21227.0,12,28186.666667,0.000000,0.076923,Extreme Greed,94.0
4,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-26,1603.1,27,17248.148148,0.444444,0.035714,Extreme Greed,79.0


## 8) Validation checks



In [10]:
print("Unique Accounts:", df["Account"].nunique())
print("Unique Dates:", df["date"].nunique())
print("Sentiment labels:", df["classification"].value_counts(dropna=False))

print("\nDate dtype:", df["date"].dtype)
print("No. of exact duplicate rows in final table:", df.duplicated().sum())


Unique Accounts: 32
Unique Dates: 479
Sentiment labels: classification
Greed            648
Fear             630
Extreme Greed    526
Neutral          376
Extreme Fear     160
Name: count, dtype: int64

Date dtype: datetime64[us]
No. of exact duplicate rows in final table: 0


## 9) Save the clean merged table

In [11]:
df.to_csv("Cleaned_Data.csv", index=False)
print("Saved: Cleaned_Data.csv")


Saved: Cleaned_Data.csv
